# Orbit: scans to high-quality 3D models (free Colab)
**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.

Every Orbit scan zip in `MyDrive/3d-scanner/scans/` that doesn't have a model yet is turned into a
photo-textured model in `MyDrive/3d-scanner/models/<name>/`:
`<name>.usdz` (tap it on iPhone to see it in AR), `<name>.glb` (web, Blender, games), `<name>.obj` + texture.

One-time setup: put this repo's `pipeline` folder at `MyDrive/3d-scanner/pipeline`. The first run builds
COLMAP and OpenMVS (about 40 minutes) and caches them in Drive; later runs start in about a minute.
Each scan then takes roughly 10 to 30 minutes.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json, shutil, subprocess
ROOT = '/content/drive/MyDrive/3d-scanner'
os.makedirs(f'{ROOT}/models', exist_ok=True)
os.makedirs(f'{ROOT}/cache', exist_ok=True)

## Tools (built once, then restored from Drive)

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq git cmake ninja-build build-essential libboost-program-options-dev libboost-graph-dev \
  libboost-system-dev libboost-iostreams-dev libboost-serialization-dev libeigen3-dev libflann-dev libfreeimage-dev \
  libmetis-dev libgoogle-glog-dev libgtest-dev libsqlite3-dev libglew-dev qtbase5-dev libqt5opengl5-dev libcgal-dev \
  libceres-dev libcurl4-openssl-dev libopencv-dev libglfw3-dev libjpeg-dev libpng-dev libtiff-dev > /dev/null
pip -q install trimesh numpy scipy networkx pillow usd-core

In [ ]:
CACHE = f'{ROOT}/cache/tools-colmap3.10-openmvs.tar.gz'
if os.path.exists(CACHE):
    !tar -xzf "{CACHE}" -C /usr/local
    print('restored COLMAP and OpenMVS from Drive')

In [ ]:
%%bash
set -e
if ! command -v colmap >/dev/null; then
  git clone -q --depth 1 --branch 3.10 https://github.com/colmap/colmap.git /content/colmap-src
  cmake -S /content/colmap-src -B /content/colmap-build -GNinja -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_BUILD_TYPE=Release > /dev/null
  ninja -C /content/colmap-build install > /dev/null
fi
if [ ! -x /usr/local/bin/OpenMVS/TextureMesh ]; then
  git clone -q https://github.com/cdcseacave/VCG.git /content/vcglib
  git clone -q --recurse-submodules https://github.com/cdcseacave/openMVS.git /content/openmvs-src
  cmake -S /content/openmvs-src -B /content/openmvs-build -DCMAKE_BUILD_TYPE=Release -DVCG_ROOT=/content/vcglib \
    -DOpenMVS_USE_CUDA=OFF -DOpenMVS_USE_PYTHON=OFF > /dev/null
  make -C /content/openmvs-build -j4 install > /dev/null
fi
colmap -h | head -1; ls /usr/local/bin/OpenMVS

In [ ]:
if not os.path.exists(CACHE):
    !tar -czf "{CACHE}" -C /usr/local bin/colmap bin/OpenMVS lib/OpenMVS share/colmap 2>/dev/null || tar -czf "{CACHE}" -C /usr/local bin/colmap bin/OpenMVS
    print('cached the tools to Drive')

## Build models for new scans

In [ ]:
!rm -rf /content/pipeline && cp -r "{ROOT}/pipeline" /content/pipeline && chmod +x /content/pipeline/*.sh
todo = []
for z in sorted(glob.glob(f'{ROOT}/scans/*.zip')):
    name = os.path.splitext(os.path.basename(z))[0]
    if not glob.glob(f'{ROOT}/models/{name}/*.usdz') and not glob.glob(f'{ROOT}/models/{name}/*.glb'):
        todo.append((name, z))
print(f'{len(todo)} scan(s) to build:', [n for n, _ in todo])

In [ ]:
for name, z in todo:
    print(f'\n===== {name} =====')
    work = f'/content/work/{name}'
    shutil.rmtree(work, ignore_errors=True)
    r = subprocess.run(['python', '/content/pipeline/prepare_scan.py', z, f'{work}/prep'], capture_output=True, text=True)
    print(r.stdout or r.stderr)
    if r.returncode: continue
    r = subprocess.run(['/content/pipeline/run_hq.sh', f'{work}/prep', f'{work}/ws', '--name', name], text=True)
    out = f'{work}/ws/out'
    if r.returncode or not os.path.isdir(out):
        print(f'{name}: failed (see the log above)'); continue
    dest = f'{ROOT}/models/{name}'
    shutil.copytree(out, dest, dirs_exist_ok=True)
    print(f'{name}: saved to {dest}:', os.listdir(dest))

Open the **Files** app on your iPhone, go to Google Drive > 3d-scanner > models, and tap the `.usdz`
to view the model in AR.